## Agent Tool Call Evaluation with MLflow

### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    openai-agents==0.22.0 \
    mcp==2.0.0 \
    databricks-mcp==0.9.2 \
    "mlflow>=3.1"

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Set up your Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Setup MLflow Tracing

In [ ]:
import mlflow
import os

# Enable auto-tracing for OpenAI
mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/Tool-Call-Evaluation")

### Define your MCP Server URLs

In [ ]:
from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient


custom_mcp_server_url = "YOUR-CUSTOM-MCP-SERVER-URL-GOES-HERE"

code_interpreter_mcp_server_url = (
    f"{workspace_host}/api/2.0/mcp/functions/"
    f"system/ai/python_exec"
)

### Define your Agent

In [ ]:
from agents import (
    Agent,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)
from agents.mcp import MCPServerStreamableHttp

@mlflow.trace
async def run_agent(user_query: str):
    # Create an OpenAI-compatible client for Databricks
    client = AsyncOpenAI(
        api_key=token,
        base_url=f"{workspace_host}/serving-endpoints"
    )

    # Configure the Databricks model
    model = OpenAIChatCompletionsModel(
        model="databricks-claude-sonnet-4-5",
        openai_client=client
    )

    async with (
        MCPServerStreamableHttp(
            name="MSLearn-MCP-Server",
            params={
                "url": custom_mcp_server_url,
                "headers": {
                    "Authorization": f"Bearer {token}"
                }
            }
        ) as custom_mcp_server,

        MCPServerStreamableHttp(
            name="code-interpreter",
            params={
                "url": code_interpreter_mcp_server_url,
                "headers": {
                    "Authorization": f"Bearer {token}"
                }
            }
        ) as code_interpreter
    ):

        agent = Agent(
            name="Multi-Tool-Agent",

            instructions="""
            You are a helpful AI assistant.

            Use the Python code interpreter for:
            - calculations
            - statistical analysis
            - numerical computation
            - executing Python code

            Use Microsoft Learn MCP for:
            - Microsoft technical documentation
            - Azure documentation
            - Azure Databricks documentation
            - Microsoft learning resources

            Do not use a tool when the question can be answered directly.
            """,

            model=model,

            mcp_servers=[
                code_interpreter,
                custom_mcp_server
            ]
        )

        result = await Runner.run(
            agent,
            user_query
        )

        return result.final_output
    

### Define the Evaluation Data

In [ ]:
eval_data = [

    {
        "inputs": {
            "query": (
                "Calculate the standard deviation of all natural "
                "numbers from 1 through 10."
            )
        }
    },

    {
        "inputs": {
            "query": (
                "Find Microsoft Learn resources for "
                "Azure Databricks."
            )
        }
    },
    
    {
        "inputs": {
            "query": "What is Python?"
        }
    }
]

### Define the Prediction Function

In [ ]:
import nest_asyncio
import asyncio

nest_asyncio.apply()

def predict_fn(query: str) -> str:
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(run_agent(query))

### Begin Evaluation

In [ ]:
from mlflow.genai.scorers import (
    ToolCallCorrectness,
    ToolCallEfficiency
)

results = mlflow.genai.evaluate(
    data=eval_data,

    predict_fn=predict_fn,

    scorers=[
        ToolCallCorrectness(),
        ToolCallEfficiency()
    ]
)

print(results.metrics)

display(results.result_df)